# H&M Participant Sampling Pipeline

Produces `train`, `val`, and `test` splits from the filtered H&M dataset.

**Pipeline:**
1. Activity floor — drop users with < `MIN_INTERACTIONS` total purchases
2. Quantile binning — partition eligible users into `N_QUANTILES` equal-frequency strata by activity
3. Proportional allocation — assign `TARGET_USERS` slots across strata
4. Top-$m_q$ selection — pick the most active users in each stratum (deterministic)
5. Random 75/25 train/test split
6. Cold-start filter — discard test rows whose user is absent from train
7. Integer encoding + 80/20 validation split


## 0. Imports


In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)


## 1. Load Raw Files


In [5]:
DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code"  # folder with the three CSVs

df          = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                          dtype={"article_id": str})
customer_df = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                          dtype={"article_id": str, "product_code": str})
article_df  = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                          dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

print(f"Raw transactions : {len(df):,}")
print(f"Unique customers : {df['customer_id'].nunique():,}")


Raw transactions : 31,788,324
Unique customers : 1,362,281


## 2. Domain Filter

Restrict to the top-6 product types, customers with > 120 unique product purchases,
and customers located in the top-5000 postal codes.


In [7]:
chosen_types = (
    article_df.groupby("product_type_name")["product_code"]
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby("customer_id")["product_code"].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df["postal_code"].value_counts()[1:5001].index
customer_df = customer_df[customer_df["postal_code"].isin(location_customers)]

chosen_articles = article_df[article_df["product_type_name"].isin(chosen_types)]
df = df[df["product_code"].isin(chosen_articles["product_code"])]
df = df[df["customer_id"].isin(chosen_customers)]
df = df[df["customer_id"].isin(customer_df["customer_id"])]

df.drop(["t_dat", "price", "article_id", "sales_channel_id"], axis=1, inplace=True)
df["bought"] = 1
df.reset_index(drop=True, inplace=True)

print(f"Filtered transactions : {len(df):,}")
print(f"Unique customers      : {df['customer_id'].nunique():,}")
print(f"Unique products       : {df['product_code'].nunique():,}")


Filtered transactions : 216,390
Unique customers      : 1,765
Unique products       : 12,782


## 3. Sampling Configuration


In [9]:
TARGET_USERS     = 1000   # number of participants to select
MIN_INTERACTIONS = 20     # activity floor: discard users below this
N_QUANTILES      = 10      # equal-frequency strata
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"


## 4. Participant Selection Function

Steps 1–4 of the pipeline: activity floor → quantile binning →
proportional allocation → top-$m_q$ deterministic selection.


In [11]:
def select_participants(df, target_users, min_interactions, n_quantiles=4,
                        user_col='customer_id', item_col='product_code'):
    """
    Deterministic quantile-based participant selection.

    1. Activity floor : discard users with s_i < min_interactions.
    2. Quantile bins  : partition eligible users into n_quantiles
                        equal-frequency strata by total interaction count.
    3. Proportional   : allocate target_users slots proportionally across
                        strata; remainder goes to highest-activity strata.
    4. Top-m_q        : within each stratum keep the top-m_q users by s_i.

    Returns the transaction row-subset for the selected customers.
    """
    # Step 1 — activity count and floor
    user_counts = (
        df.groupby(user_col)[item_col]
        .count()
        .rename('n_interactions')
        .reset_index()
    )
    eligible = user_counts[
        user_counts['n_interactions'] >= min_interactions
    ].copy().reset_index(drop=True)

    print(f"Step 1 — activity floor (>= {min_interactions}):")
    print(f"  Total customers  : {len(user_counts)}")
    print(f"  Eligible         : {len(eligible)}")
    print(f"  Dropped          : {len(user_counts) - len(eligible)}")

    # Step 2 — equal-frequency quantile binning
    eligible['quantile_bin'] = pd.qcut(
        eligible['n_interactions'], q=n_quantiles,
        labels=False, duplicates='drop'
    )
    actual_bins = sorted(eligible['quantile_bin'].unique())
    bin_sizes   = eligible.groupby('quantile_bin').size()

    print(f"\nStep 2 — quantile binning ({n_quantiles} equal-frequency strata):")
    for b in actual_bins:
        grp = eligible[eligible['quantile_bin'] == b]['n_interactions']
        print(f"  Bin {b}: [{grp.min():4d}, {grp.max():4d}]  n={int(bin_sizes[b])}")

    # Step 3 — proportional allocation
    total_eligible = len(eligible)
    alloc = {
        b: max(1, int(target_users * bin_sizes[b] / total_eligible))
        for b in actual_bins
    }
    remainder = target_users - sum(alloc.values())
    for b in sorted(actual_bins, reverse=True):
        if remainder <= 0:
            break
        add = min(remainder, int(bin_sizes[b]) - alloc[b])
        alloc[b] += add
        remainder -= add

    print(f"\nStep 3 — proportional allocation (target={target_users}):")
    for b in actual_bins:
        print(f"  Bin {b}: allocated {alloc[b]}")

    # Step 4 — top-m_q selection (deterministic)
    selected_rows = []
    for b in actual_bins:
        grp = (
            eligible[eligible['quantile_bin'] == b]
            .sort_values('n_interactions', ascending=False)
        )
        selected_rows.append(grp.head(alloc[b]))

    selected_ids = pd.concat(selected_rows, ignore_index=True)[user_col].values
    print(f"\nStep 4 — top-m_q selection: {len(selected_ids)} users selected")

    df_sub = df[df[user_col].isin(selected_ids)].copy()
    df_sub.reset_index(drop=True, inplace=True)
    return df_sub


## 5. Run Pipeline

Applies participant selection, then the 75/25 train/test split,
cold-start user filter, integer encoding, and 80/20 val split.
Results are cached to disk; subsequent runs reload from cache.


In [13]:
cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_q{N_QUANTILES}"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

if all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]):
    # ── Fast path: reload from disk ─────────────────────────────────────
    print(f"Loading cached dataset: {cache_tag}")
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    meta_df  = pd.read_csv(meta_path)
    N_USERS  = int(meta_df.loc[meta_df["key"] == "n_users",  "value"].iloc[0])
    N_ITEMS  = int(meta_df.loc[meta_df["key"] == "n_items",  "value"].iloc[0])
    print(f"  Users : {N_USERS} | Items : {N_ITEMS}")
    print(f"  Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

else:
    # ── Slow path: run pipeline ──────────────────────────────────────────
    print(f"Cache not found — running sampling pipeline ({cache_tag}) ...\n")

    # Steps 1–4: participant selection
    df_sampled = select_participants(
        df,
        target_users=TARGET_USERS,
        min_interactions=MIN_INTERACTIONS,
        n_quantiles=N_QUANTILES,
    )
    print(f"\nSelected rows : {len(df_sampled):,}")

    # Step 5: random 75/25 train/test split
    X = df_sampled[["customer_id", "product_code"]].values
    y = df_sampled["bought"].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=SEED
    )
    train_df = pd.DataFrame(X_train, columns=["customer_id", "product_code"])
    test_df  = pd.DataFrame(X_test,  columns=["customer_id", "product_code"])
    train_df["bought"] = y_train
    test_df["bought"]  = y_test
    print(f"\nStep 5 — 75/25 split:")
    print(f"  Train raw : {len(train_df):,} | Test raw : {len(test_df):,}")

    # Step 6: cold-start filter — test users must appear in train
    train_users = set(train_df["customer_id"])
    mask        = test_df["customer_id"].isin(train_users)
    n_dropped   = (~mask).sum()
    test_df     = test_df[mask].copy()
    print(f"\nStep 6 — cold-start filter:")
    print(f"  Dropped {n_dropped} test rows with users absent from train")
    print(f"  Test after filter : {len(test_df):,}")

    # Step 7a: integer encoding (fit on combined train + test)
    all_users = np.unique(
        pd.concat([train_df[["customer_id"]], test_df[["customer_id"]]])["customer_id"]
    )
    all_items = np.unique(
        pd.concat([train_df[["product_code"]], test_df[["product_code"]]])["product_code"]
    )
    customer_id2idx = {c: i for i, c in enumerate(all_users)}
    item_id2idx     = {a: i for i, a in enumerate(all_items)}
    for df_ in [train_df, test_df]:
        df_["customer_id"]  = df_["customer_id"].map(customer_id2idx)
        df_["product_code"] = df_["product_code"].map(item_id2idx)

    N_USERS = len(all_users)
    N_ITEMS = len(all_items)
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # Step 7b: 80/20 validation split of train
    n_val    = int(len(train_df) * 0.2)
    val_df   = train_df.tail(n_val).reset_index(drop=True)
    train_df = train_df.head(len(train_df) - n_val).reset_index(drop=True)

    # Save to disk
    os.makedirs(SAMPLED_DATA_DIR, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path,     index=False)
    test_df.to_csv(test_path,   index=False)
    pd.DataFrame([
        {"key": "n_users",          "value": N_USERS},
        {"key": "n_items",          "value": N_ITEMS},
        {"key": "n_train",          "value": len(train_df)},
        {"key": "n_val",            "value": len(val_df)},
        {"key": "n_test",           "value": len(test_df)},
        {"key": "target_users",     "value": TARGET_USERS},
        {"key": "min_interactions", "value": MIN_INTERACTIONS},
        {"key": "n_quantiles",      "value": N_QUANTILES},
        {"key": "seed",             "value": SEED},
    ]).to_csv(meta_path, index=False)

    density = len(train_df) / (N_USERS * N_ITEMS) * 100
    print(f"\nTotal selected: {N_USERS} users")
    print(f"Dropped {n_dropped} test rows with users absent from train")
    print(f"Saved to {SAMPLED_DATA_DIR}/")
    print(f"Users   : {N_USERS}")
    print(f"Items   : {N_ITEMS}")
    print(f"Train   : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Density : {density:.4f}%")


Cache not found — running sampling pipeline (u1000_m20_q10) ...

Step 1 — activity floor (>= 20):
  Total customers  : 1765
  Eligible         : 1765
  Dropped          : 0

Step 2 — quantile binning (10 equal-frequency strata):
  Bin 0: [  32,   69]  n=179
  Bin 1: [  70,   81]  n=193
  Bin 2: [  82,   90]  n=168
  Bin 3: [  91,  100]  n=179
  Bin 4: [ 101,  110]  n=176
  Bin 5: [ 111,  121]  n=169
  Bin 6: [ 122,  135]  n=172
  Bin 7: [ 136,  154]  n=183
  Bin 8: [ 155,  188]  n=169
  Bin 9: [ 189,  528]  n=177

Step 3 — proportional allocation (target=1000):
  Bin 0: allocated 101
  Bin 1: allocated 109
  Bin 2: allocated 95
  Bin 3: allocated 101
  Bin 4: allocated 99
  Bin 5: allocated 95
  Bin 6: allocated 97
  Bin 7: allocated 103
  Bin 8: allocated 95
  Bin 9: allocated 105

Step 4 — top-m_q selection: 1000 users selected

Selected rows : 130,102

Step 5 — 75/25 split:
  Train raw : 97,576 | Test raw : 32,526

Step 6 — cold-start filter:
  Dropped 0 test rows with users absent 

## 6. Verification


In [15]:
# Confirm: every test user appears in train
train_users = set(train_df["customer_id"].unique())
test_users  = set(test_df["customer_id"].unique())
cold_start  = test_users - train_users
print(f"Train users          : {len(train_users)}")
print(f"Test users           : {len(test_users)}")
print(f"Cold-start test users: {len(cold_start)}  (should be 0)")
assert len(cold_start) == 0, "Cold-start users found in test set!"

# Interaction count distribution across train users
counts = train_df.groupby("customer_id")["product_code"].count()
print(f"\nTrain interactions per user:")
print(f"  Min    : {counts.min()}")
print(f"  Median : {counts.median():.0f}")
print(f"  Max    : {counts.max()}")
print(f"  Mean   : {counts.mean():.1f}")


Train users          : 1000
Test users           : 1000
Cold-start test users: 0  (should be 0)

Train interactions per user:
  Min    : 28
  Median : 68
  Max    : 316
  Mean   : 78.1
